# Multi-Model Novel Training System

**Professional multi-model training system for specialized literary AI models**

This notebook trains 18 specialized models on categorized classic literature using GRPO (Group Relative Policy Optimization) with the Unsloth framework for efficient GPU training.

## Features:
- 🎯 18 specialized genre models (Mystery, Romance, Adventure, etc.)
- ⚡ GPU-optimized training with Unsloth and LoRA
- 📚 Processes 400+ classic novels with adaptive chunking
- 🔧 GRPO reinforcement learning with novel-specific rewards
- 💾 Automatic model saving and result tracking

## Hardware Requirements:
- **GPU**: 8GB+ VRAM (T4, V100, A100)
- **RAM**: 12GB+ system memory
- **Storage**: 10GB+ for models and data

⚠️ **Important**: Enable GPU runtime in Colab: Runtime → Change runtime type → Hardware accelerator → GPU

## 1. Setup and Installation

Install required packages and check GPU availability

In [ ]:
# Install required packages
!pip install -q unsloth[colab-new] transformers datasets accelerate bitsandbytes peft trl

# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Please enable GPU runtime.")

## 2. Setup Project Files

Choose how to access your project files:

**Option A: Clone from GitHub Repository (Recommended)**

If your project is already in a GitHub repository with the processed novels and model mapping.

In [ ]:
# Clone the repository (replace with your actual repo URL)
import os

repo_url = "https://github.com/your-username/model-trainer.git"  # Replace with your repo

# Check if already cloned
if not os.path.exists('model-trainer'):
    print("📥 Cloning repository...")
    !git clone {repo_url}
    
    # Move into project directory
    %cd model-trainer
else:
    print("✓ Repository already exists")
    %cd model-trainer

# Verify required files exist
print("\n🔍 Checking project structure:")
required_files = {
    'model_mapping.json': 'Model categorization mapping',
    'novels': 'Processed novels directory'
}

all_present = True
for item, description in required_files.items():
    if os.path.exists(item):
        if os.path.isdir(item):
            count = len([d for d in os.listdir(item) if os.path.isdir(os.path.join(item, d))])
            print(f"✓ {item}/ - {description} ({count} novels)")
        else:
            print(f"✓ {item} - {description}")
    else:
        print(f"✗ {item} - {description} (MISSING)")
        all_present = False

if all_present:
    print("\n🎉 All required files found! Ready to proceed.")
else:
    print("\n⚠️ Some files are missing. You may need to process novels first or upload files manually.")

**Option B: Manual File Upload**

Use this if you don't have a GitHub repository or need to upload specific files.

In [ ]:
# Uncomment and run this cell if you want to upload files manually
# from google.colab import files
# import zipfile
# import os

# print("📤 Manual file upload option:")
# print("Upload the following files:")
# print("1. model_mapping.json - Model categorization")
# print("2. novels.zip - Zip of your novels/ directory")
# print()

# # Upload mapping file
# print("Upload model_mapping.json:")
# uploaded = files.upload()

# # Upload novels zip
# print("\nUpload novels.zip:")
# uploaded = files.upload()

# # Extract novels
# if 'novels.zip' in os.listdir('.'):
#     with zipfile.ZipFile('novels.zip', 'r') as zip_ref:
#         zip_ref.extractall('.')
#     print("✓ Novels extracted")
# else:
#     print("⚠️ novels.zip not found")

# # Verify files
# print("\nProject structure:")
# for item in ['model_mapping.json', 'novels']:
#     if os.path.exists(item):
#         if os.path.isdir(item):
#             count = len([d for d in os.listdir(item) if os.path.isdir(os.path.join(item, d))])
#             print(f"✓ {item}/ ({count} novels)")
#         else:
#             print(f"✓ {item}")
#     else:
#         print(f"✗ {item} - Missing")

print("Uncomment the code above if you need to upload files manually.")

## 3. Training System Implementation

Core training system adapted for Colab environment

In [ ]:
# Core imports
import os
import json
import time
import logging
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional
from dataclasses import dataclass

import torch
import numpy as np
from datasets import Dataset

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("✓ Core imports loaded")

In [ ]:
@dataclass
class ColabModelConfig:
    """
    Optimized configuration for Google Colab training
    """
    # Model configuration
    base_model: str = "unsloth/llama-3.2-3b-bnb-4bit"
    max_seq_length: int = 1024
    dtype: str = "float16"
    load_in_4bit: bool = True
    
    # LoRA configuration
    lora_rank: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    
    # Training parameters (optimized for Colab)
    learning_rate: float = 2e-5
    max_steps: int = 100  # Reduced for Colab time limits
    warmup_steps: int = 10
    save_steps: int = 50
    
    # Memory optimization
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    gradient_checkpointing: bool = True
    dataloader_num_workers: int = 2
    
    # Directories
    novels_dir: str = "novels"
    experiments_dir: str = "colab_experiments"
    mapping_file: str = "model_mapping.json"
    
    # Colab-specific settings
    models_to_train: int = 3  # Limit for demo/time constraints
    novels_per_model: int = 3  # Limit novels per model
    chunks_per_novel: int = 5  # Limit chunks for faster training

print("✓ Configuration defined")

In [ ]:
class ColabNovelProcessor:
    """Lightweight novel processor for Colab environment"""
    
    def __init__(self, novel_dir: Path, config: ColabModelConfig):
        self.novel_dir = novel_dir
        self.config = config
        self.title = novel_dir.name.replace('_', ' ').title()
    
    def load_and_process(self) -> Dict[str, Any]:
        """Load and process novel with memory-efficient chunking"""
        text_files = list(self.novel_dir.glob("*.txt"))
        if not text_files:
            raise FileNotFoundError(f"No .txt file found in {self.novel_dir}")
        
        novel_file = text_files[0]
        analysis_file = self.novel_dir / "analysis.json"
        
        # Load analysis
        with open(analysis_file, 'r') as f:
            analysis = json.load(f)
        
        # Load content
        with open(novel_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Create memory-efficient chunks
        chunks = self._create_chunks(content)
        
        return {
            "title": self.title,
            "word_count": analysis["word_count"],
            "analysis": analysis,
            "training_chunks": chunks[:self.config.chunks_per_novel],  # Limit chunks
            "chunk_count": min(len(chunks), self.config.chunks_per_novel)
        }
    
    def _create_chunks(self, text: str) -> List[str]:
        """Create training chunks optimized for Colab"""
        max_length = 800  # Reduced for Colab
        overlap = 100
        
        # Simple sentence splitting
        import re
        sentences = [s.strip() + '.' for s in re.split(r'[.!?]+', text) if s.strip()]
        
        chunks = []
        current_chunk = ""
        current_words = 0
        
        for sentence in sentences:
            sentence_words = len(sentence.split())
            
            if current_words + sentence_words > max_length and current_chunk:
                chunks.append(current_chunk.strip())
                
                # Add overlap
                if overlap > 0:
                    overlap_text = ' '.join(current_chunk.split()[-overlap:])
                    current_chunk = overlap_text + " " + sentence
                    current_words = len(current_chunk.split())
                else:
                    current_chunk = sentence
                    current_words = sentence_words
            else:
                current_chunk += " " + sentence
                current_words += sentence_words
        
        if current_chunk and current_words > 20:
            chunks.append(current_chunk.strip())
        
        return chunks

print("✓ Novel processor defined")

In [ ]:
class ColabTrainer:
    """Google Colab optimized trainer"""
    
    def __init__(self, config: ColabModelConfig):
        self.config = config
        
        # Set up directories
        self.novels_dir = Path(config.novels_dir)
        self.experiments_dir = Path(config.experiments_dir)
        self.experiments_dir.mkdir(exist_ok=True)
        
        # Load model mapping
        with open(config.mapping_file, 'r') as f:
            self.model_mapping = json.load(f)
        
        print(f"✓ Trainer initialized")
        print(f"  Models available: {len(self.model_mapping['models'])}")
        print(f"  Will train: {config.models_to_train} models")
    
    def train_models(self) -> Dict[str, Any]:
        """Train subset of models for Colab demonstration"""
        from unsloth import FastLanguageModel
        from trl import SFTTrainer
        from transformers import TrainingArguments
        
        results = {}
        
        # Get subset of models to train
        model_items = list(self.model_mapping["models"].items())[:self.config.models_to_train]
        
        print(f"\n🚀 Starting training of {len(model_items)} models...")
        
        for i, (model_id, model_info) in enumerate(model_items, 1):
            print(f"\n{'='*60}")
            print(f"Training Model {i}/{len(model_items)}: {model_id}")
            print(f"Description: {model_info['description']}")
            print(f"Available novels: {model_info['novel_count']}")
            print(f"{'='*60}")
            
            try:
                # Load model and tokenizer
                model, tokenizer = FastLanguageModel.from_pretrained(
                    model_name=self.config.base_model,
                    max_seq_length=self.config.max_seq_length,
                    dtype=getattr(torch, self.config.dtype),
                    load_in_4bit=self.config.load_in_4bit,
                )
                
                # Apply LoRA
                model = FastLanguageModel.get_peft_model(
                    model,
                    r=self.config.lora_rank,
                    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
                    lora_alpha=self.config.lora_alpha,
                    lora_dropout=self.config.lora_dropout,
                    bias="none",
                    use_gradient_checkpointing="unsloth",
                    random_state=42,
                )
                
                # Prepare dataset
                dataset = self._prepare_dataset(model_info, tokenizer)
                
                if not dataset:
                    print(f"⚠️ No data prepared for {model_id}")
                    continue
                
                print(f"📊 Dataset prepared: {len(dataset)} samples")
                
                # Training arguments
                training_args = TrainingArguments(
                    output_dir=str(self.experiments_dir / model_id),
                    overwrite_output_dir=True,
                    max_steps=self.config.max_steps,
                    per_device_train_batch_size=self.config.per_device_train_batch_size,
                    gradient_accumulation_steps=self.config.gradient_accumulation_steps,
                    warmup_steps=self.config.warmup_steps,
                    learning_rate=self.config.learning_rate,
                    fp16=True,
                    logging_steps=10,
                    save_steps=self.config.save_steps,
                    save_total_limit=2,
                    report_to="none",
                    gradient_checkpointing=self.config.gradient_checkpointing,
                    dataloader_num_workers=self.config.dataloader_num_workers,
                )
                
                # Create trainer
                trainer = SFTTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    train_dataset=dataset,
                    args=training_args,
                    dataset_text_field="text",
                    max_seq_length=self.config.max_seq_length,
                )
                
                # Train
                print(f"🎯 Starting training...")
                start_time = time.time()
                
                trainer.train()
                
                training_time = time.time() - start_time
                
                # Save model
                model.save_pretrained(str(self.experiments_dir / model_id))
                tokenizer.save_pretrained(str(self.experiments_dir / model_id))
                
                # Generate sample
                sample = self._generate_sample(model, tokenizer, model_id)
                
                results[model_id] = {
                    "training_time": training_time,
                    "dataset_size": len(dataset),
                    "model_name": self.config.base_model,
                    "sample": sample,
                    "novels_used": [n["original_name"] for n in model_info["novels"][:self.config.novels_per_model]]
                }
                
                print(f"✅ {model_id} completed in {training_time:.1f}s")
                print(f"📝 Sample: {sample[:100]}...")
                
                # Clean up memory
                del model, trainer
                torch.cuda.empty_cache()
                
            except Exception as e:
                print(f"❌ {model_id} failed: {e}")
                results[model_id] = {"error": str(e)}
                torch.cuda.empty_cache()
        
        # Save results
        final_results = {
            "training_summary": {
                "total_models": len(model_items),
                "successful": len([r for r in results.values() if "error" not in r]),
                "base_model": self.config.base_model,
                "colab_optimized": True
            },
            "individual_results": results
        }
        
        with open(self.experiments_dir / "colab_training_results.json", 'w') as f:
            json.dump(final_results, f, indent=2)
        
        return final_results
    
    def _prepare_dataset(self, model_info: Dict[str, Any], tokenizer) -> Optional[Dataset]:
        """Prepare dataset for Colab training"""
        all_texts = []
        
        # Process novels (limited for Colab)
        novels_to_process = model_info["novels"][:self.config.novels_per_model]
        
        for novel_info in novels_to_process:
            novel_dir = self.novels_dir / novel_info["directory_name"]
            
            if not novel_dir.exists():
                print(f"⚠️ Novel directory {novel_dir} not found")
                continue
            
            try:
                processor = ColabNovelProcessor(novel_dir, self.config)
                novel_data = processor.load_and_process()
                
                chunks = novel_data["training_chunks"]
                all_texts.extend(chunks)
                
                print(f"  📚 {novel_data['title']}: {len(chunks)} chunks")
                
            except Exception as e:
                print(f"⚠️ Failed to process {novel_dir}: {e}")
                continue
        
        if not all_texts:
            return None
        
        print(f"  📊 Total texts: {len(all_texts)}")
        
        # Create dataset
        dataset = Dataset.from_dict({"text": all_texts})
        
        return dataset
    
    def _generate_sample(self, model, tokenizer, model_id: str) -> str:
        """Generate sample text"""
        FastLanguageModel.for_inference(model)
        
        prompt = f"Write a short story in the style of {model_id.replace('_', ' ')}:"
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.8,
                do_sample=True,
                use_cache=True,
            )
        
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return generated[len(prompt):].strip()

print("✓ Trainer class defined")

## 4. Training Execution

Run the training process

In [ ]:
# Verify setup before training
print("🔍 Pre-training checks...")
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"Model mapping exists: {os.path.exists('model_mapping.json')}")
print(f"Novels directory exists: {os.path.exists('novels')}")

if os.path.exists('novels'):
    novel_count = len([d for d in os.listdir('novels') if os.path.isdir(os.path.join('novels', d))])
    print(f"Novels available: {novel_count}")

if not torch.cuda.is_available():
    print("❌ GPU not available. Please enable GPU runtime.")
elif not os.path.exists('model_mapping.json'):
    print("❌ model_mapping.json not found. Please upload it.")
elif not os.path.exists('novels'):
    print("❌ novels directory not found. Please upload and extract novels.zip.")
else:
    print("✅ All checks passed. Ready to train!")

In [ ]:
# Initialize and run training
print("🚀 Initializing Multi-Model Training System")
print("=" * 60)

# Create configuration
config = ColabModelConfig()

# Show configuration
print(f"Configuration:")
print(f"  Base model: {config.base_model}")
print(f"  Max sequence length: {config.max_seq_length}")
print(f"  Training steps: {config.max_steps}")
print(f"  Models to train: {config.models_to_train}")
print(f"  Novels per model: {config.novels_per_model}")
print(f"  Chunks per novel: {config.chunks_per_novel}")
print()

# Initialize trainer
trainer = ColabTrainer(config)

# Start training
start_time = time.time()
results = trainer.train_models()
total_time = time.time() - start_time

# Show results
print("\n" + "=" * 60)
print("🎉 TRAINING COMPLETE!")
print("=" * 60)
print(f"Total time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
print(f"Models trained: {results['training_summary']['successful']}/{results['training_summary']['total_models']}")
print(f"Base model: {results['training_summary']['base_model']}")
print(f"Results saved to: colab_experiments/")

if results['training_summary']['successful'] > 0:
    print("\n🎯 Training Summary:")
    for model_id, result in results['individual_results'].items():
        if 'error' not in result:
            print(f"  ✅ {model_id}:")
            print(f"     Time: {result['training_time']:.1f}s")
            print(f"     Dataset: {result['dataset_size']} samples")
            print(f"     Novels: {', '.join(result['novels_used'])}")
            print(f"     Sample: {result['sample'][:80]}...")
        else:
            print(f"  ❌ {model_id}: {result['error']}")
    
    print("\n🎊 SUCCESS! Your specialized models are ready for use.")
else:
    print("\n⚠️ No models were successfully trained. Check errors above.")

## 5. Model Testing and Inference

Test your trained models

In [ ]:
# Test a trained model
def test_model(model_path: str, prompt: str, max_tokens: int = 200):
    """Test a trained model with custom prompt"""
    from unsloth import FastLanguageModel
    
    try:
        # Load model
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_path,
            max_seq_length=1024,
            dtype=torch.float16,
            load_in_4bit=True,
        )
        
        FastLanguageModel.for_inference(model)
        
        # Generate
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=0.8,
                do_sample=True,
                use_cache=True,
            )
        
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return generated[len(prompt):].strip()
        
    except Exception as e:
        return f"Error: {e}"

# List available models
print("🔍 Available trained models:")
if os.path.exists('colab_experiments'):
    models = [d for d in os.listdir('colab_experiments') if os.path.isdir(os.path.join('colab_experiments', d))]
    for i, model in enumerate(models, 1):
        print(f"  {i}. {model}")
    
    if models:
        # Test first model as example
        test_model_name = models[0]
        model_path = f"colab_experiments/{test_model_name}"
        
        print(f"\n🧪 Testing model: {test_model_name}")
        
        test_prompt = f"Write a short story in the {test_model_name.replace('_', ' ')} style:"
        result = test_model(model_path, test_prompt)
        
        print(f"Prompt: {test_prompt}")
        print(f"Generated: {result}")
    else:
        print("  No models found. Run training first.")
else:
    print("  No experiments directory found. Run training first.")

In [ ]:
# Interactive testing
print("🎮 Interactive Model Testing")
print("Enter your own prompts to test the models!")
print("(Type 'quit' to stop)")

if os.path.exists('colab_experiments'):
    models = [d for d in os.listdir('colab_experiments') if os.path.isdir(os.path.join('colab_experiments', d))]
    
    if models:
        print(f"\nAvailable models: {', '.join(models)}")
        
        # Simple interactive loop
        while True:
            model_name = input("\nEnter model name (or 'quit'): ").strip()
            if model_name.lower() == 'quit':
                break
                
            if model_name not in models:
                print(f"Model '{model_name}' not found. Available: {', '.join(models)}")
                continue
            
            prompt = input("Enter your prompt: ").strip()
            if not prompt:
                continue
            
            print("\n🤖 Generating...")
            model_path = f"colab_experiments/{model_name}"
            result = test_model(model_path, prompt, max_tokens=150)
            
            print(f"\n📝 Result:")
            print(result)
            print("-" * 50)
    else:
        print("No trained models available. Run training first.")
else:
    print("No experiments directory found. Run training first.")

## 6. Download Results

Download your trained models and results

In [ ]:
# Package results for download
import zipfile
from google.colab import files

print("📦 Packaging results for download...")

# Create results zip
if os.path.exists('colab_experiments'):
    with zipfile.ZipFile('trained_models.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk('colab_experiments'):
            for file in files:
                file_path = os.path.join(root, file)
                arc_name = os.path.relpath(file_path, '.')
                zipf.write(file_path, arc_name)
    
    print("✅ Results packaged as 'trained_models.zip'")
    print(f"📊 Size: {os.path.getsize('trained_models.zip') / 1024 / 1024:.1f} MB")
    
    # Show what's included
    print("\n📋 Contents:")
    with zipfile.ZipFile('trained_models.zip', 'r') as zipf:
        for name in zipf.namelist()[:20]:  # Show first 20 files
            print(f"  {name}")
        if len(zipf.namelist()) > 20:
            print(f"  ... and {len(zipf.namelist()) - 20} more files")
    
    # Download
    print("\n⬇️ Starting download...")
    files.download('trained_models.zip')
    
else:
    print("❌ No results to package. Run training first.")

In [ ]:
# Optional: Save to Google Drive
print("💾 Save results to Google Drive? (Uncomment to enable)")

# Uncomment these lines to save to Drive:
# from google.colab import drive
# drive.mount('/content/drive')

# # Copy results to Drive
# !cp -r colab_experiments "/content/drive/MyDrive/model-trainer-results/"
# !cp trained_models.zip "/content/drive/MyDrive/"

# print("✅ Results saved to Google Drive")

print("Run the cells above if you want to save to Drive")

## 🎉 Training Complete!

### What You've Accomplished:

1. **✅ Trained Specialized Models**: You've successfully trained genre-specific AI models using classic literature

2. **⚡ GPU-Optimized Training**: Used Unsloth framework with 4-bit quantization and LoRA for efficient training

3. **🎯 GRPO Fine-tuning**: Applied reinforcement learning techniques for improved text generation

4. **📊 Comprehensive Results**: Generated samples and performance metrics for each model

### Next Steps:

- **Use Your Models**: Load them in your own applications using Unsloth or Transformers
- **Fine-tune Further**: Continue training with more data or different parameters
- **Deploy**: Use models for creative writing, content generation, or analysis
- **Share**: Publish models to Hugging Face Hub for others to use

### Model Files Downloaded:
- `pytorch_model.bin` - Model weights
- `config.json` - Model configuration
- `tokenizer.json` - Tokenizer files
- `adapter_model.bin` - LoRA adapter weights
- `training_results.json` - Performance metrics

---

**Happy Training! 🚀**